# AgriCare AI — Notebook 05: MobileNetV2 Transfer Learning Architecture

**Project:** AgriCare AI (LeafAgro)  
**Phase:** Phase 37 — Model Architecture Design & Validation  

---
### Overview
This notebook walks through the MobileNetV2 transfer learning architecture designed for edge plant disease classification. It covers:
1. Loading the 19 verified agricultural classes
2. Constructing the MobileNetV2 backbone and lightweight classifier head
3. Configuring Stage A (Feature Extraction) and Stage B (Fine-Tuning)
4. Running forward passes with synthetic tensors
5. Validating softmax probability outputs and serialization parity

> **Important:** In accordance with Phase 37 restrictions, this notebook **does not perform model training**.

## 1. Environment & Module Imports
We import TensorFlow and our project modules (`config`, `model`, `model_utils`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf

# Ensure project src is in Python path
notebook_dir = Path.cwd()
ai_model_dir = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir / "ai_model"
if str(ai_model_dir) not in sys.path:
    sys.path.insert(0, str(ai_model_dir))

from src.config import (
    IMG_HEIGHT, IMG_WIDTH, CHANNELS, IMAGE_SIZE,
    NUM_CLASSES, CLASSES, DROPOUT_RATE, LEARNING_RATE,
    FINE_TUNE_LEARNING_RATE, FINE_TUNE_AT_LAYER,
    LABELS_PATH, MODEL_ARCHITECTURE_PATH
)
from src.model import (
    build_model, build_feature_extraction_model,
    configure_fine_tuning, get_model_summary,
    validate_model_output
)
from src.model_utils import validate_labels_file

print(f"TensorFlow Version: {tf.__version__}")
print(f"Target Image Shape: ({IMG_HEIGHT}, {IMG_WIDTH}, {CHANNELS})")
print(f"Target Class Count: {NUM_CLASSES}")

## 2. Verified Class Labels & Index Mapping
Let us load and inspect `ai_model/models/labels.json` to verify the 19 agricultural disease categories.

In [ ]:
labels_status = validate_labels_file(LABELS_PATH)
print(f"Labels validation: {labels_status['status']}")
print(f"Total verified classes: {labels_status['num_classes']}")

for idx, cls_name in enumerate(labels_status["class_names"]):
    print(f"  [{idx:02d}] {cls_name}")

## 3. Building the MobileNetV2 Architecture
We construct the model using `build_model()`. Notice the embedded `layers.Rescaling(1/127.5, offset=-1.0)` layer that normalizes raw [0, 255] pixel inputs internally.

In [ ]:
model, base_model = build_model(num_classes=NUM_CLASSES, dropout_rate=DROPOUT_RATE)
print(f"Model Name: {model.name}")
print(f"Input Shape: {model.input_shape}")
print(f"Output Shape: {model.output_shape}")
print(f"Total Layers: {len(model.layers)}")
print(f"Backbone Layers: {len(base_model.layers)}")

## 4. Stage A — Feature Extraction Configuration
In Stage A, the MobileNetV2 backbone is frozen (`trainable = False`), and only the classifier head is trained with an initial learning rate of `0.001`.

In [ ]:
stage_a_model, stage_a_base = build_feature_extraction_model(
    num_classes=NUM_CLASSES,
    learning_rate=LEARNING_RATE
)

total_params = stage_a_model.count_params()
trainable_a = sum(w.numpy().size for w in stage_a_model.trainable_weights)
non_trainable_a = sum(w.numpy().size for w in stage_a_model.non_trainable_weights)

print("Stage A Parameter Allocation:")
print(f"  Total Parameters:        {total_params:,}")
print(f"  Trainable (Head only):   {trainable_a:,}")
print(f"  Non-trainable (Backbone):{non_trainable_a:,}")

## 5. Stage B — Fine-Tuning Configuration
In Stage B, we unfreeze upper convolutional blocks (layer index >= 100) while keeping `BatchNormalization` layers frozen to preserve running statistics, dropping the learning rate to `1e-5`.

In [ ]:
stage_b_model = configure_fine_tuning(
    stage_a_model,
    stage_a_base,
    fine_tune_at=FINE_TUNE_AT_LAYER,
    learning_rate=FINE_TUNE_LEARNING_RATE
)

trainable_b = sum(w.numpy().size for w in stage_b_model.trainable_weights)
non_trainable_b = sum(w.numpy().size for w in stage_b_model.non_trainable_weights)

print("Stage B Parameter Allocation:")
print(f"  Total Parameters:          {total_params:,}")
print(f"  Trainable (Upper + Head):  {trainable_b:,}")
print(f"  Non-trainable (Lower + BN):{non_trainable_b:,}")

## 6. Architecture Summary Table
Let us inspect the complete layer connectivity of the model.

In [ ]:
stage_a_model.summary()

## 7. Dummy Forward Pass & Softmax Probability Verification
We pass a synthetic batch of 4 RGB images through the un-trained model to ensure the computational graph executes smoothly without NaNs or Infinities.

In [ ]:
dummy_batch = tf.random.uniform((4, IMG_HEIGHT, IMG_WIDTH, CHANNELS), minval=0.0, maxval=255.0, seed=42)
predictions = stage_a_model(dummy_batch, training=False)

print(f"Batch Predictions Shape: {predictions.shape}")
print(f"Any NaNs present: {bool(tf.reduce_any(tf.math.is_nan(predictions)).numpy())}")
print(f"Any Infs present: {bool(tf.reduce_any(tf.math.is_inf(predictions)).numpy())}")

# Verify probability axioms: sum(p_i) == 1.0
row_sums = tf.reduce_sum(predictions, axis=-1).numpy()
for sample_idx, s in enumerate(row_sums):
    print(f"  Sample {sample_idx} Softmax Sum: {s:.6f} (Valid: {abs(s - 1.0) < 1e-5})")

## 8. Model Architecture Serialization Parity Test
We test saving the model to a temporary file and reloading it to guarantee serialization compatibility for future Phase 38 training and TensorFlow Lite export.

In [ ]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmp_dir:
    test_save_path = os.path.join(tmp_dir, "arch_test.keras")
    stage_a_model.save(test_save_path)
    print(f"Model saved to temporary path: {test_save_path}")
    
    reloaded = tf.keras.models.load_model(test_save_path)
    reloaded_preds = reloaded(dummy_batch, training=False)
    
    max_diff = np.max(np.abs(predictions.numpy() - reloaded_preds.numpy()))
    print(f"Reload parity check — Maximum absolute output difference: {max_diff:.8e}")
    assert max_diff < 1e-5, "Reloaded predictions diverge from original model!"
    print("Parity check PASSED successfully!")